In [15]:
import json
import pandas as pd
import argparse
from tqdm import tqdm
from langchain import PromptTemplate, LLMChain
from langchain.llms.ollama import Ollama
from langchain.output_parsers import PydanticOutputParser
from langchain.schema import OutputParserException
from pydantic import BaseModel, Field
from typing import List
import re

In [10]:
# Load the JSON file containing the list of jobs
with open("./data/Final_dataset.json", "r", encoding="utf-8") as file:
    jobs_list = json.load(file)

# If you need to assign an ID to each job (if not already present), uncomment below:
for index, job in enumerate(jobs_list, start=1):
    job["id"] = index


In [14]:
# Define the schema for each skill and for the overall extraction for a job.
class Skill(BaseModel):
    skill: str
    influence: int = Field(..., ge=0, le=100)  # Influence percentage (0-100)

class JobSkills(BaseModel):
    soft_skills: List[Skill]
    hard_skills: List[Skill]

# Create an output parser using the defined Pydantic model.
output_parser = PydanticOutputParser(pydantic_object=JobSkills)
# Get the JSON schema instructions and escape curly braces for safe formatting.
format_instructions = output_parser.get_format_instructions()
escaped_format_instructions = format_instructions.replace("{", "{{").replace("}", "}}")

# Create the prompt template with the escaped format instructions.
prompt_template = f"""
You are given the job description text from a job posting. Your task is to extract the soft skills and hard skills required for the job mentioned in the job description.
Soft skills include communication, teamwork, adaptability, problem-solving, leadership, emotional intelligence, and time management.
Hard skills include the following:
- Programming and Software Development
- Data Analysis and Statistical Analysis
- Project Management
- Financial Analysis and Forecasting
- Technical Writing and Documentation
- Machine Learning and Artificial Intelligence
- Graphic Design and Visual Communication
- Digital Marketing and SEO/SEM
- Web Development
- Database Management and SQL
- Cybersecurity and Information Security
- IT Networking and Infrastructure Management
- Quality Assurance and Software Testing
- Computer-Aided Design (CAD) and 3D Modeling
- Engineering Design and Simulation
- Scientific Research and Laboratory Skills
- Legal Research and Compliance
- Social Media Management and Analytics
- Content Creation and Copywriting
- Multimedia Production and Video Editing
- Technical Support and Troubleshooting
- Operating Systems Administration
- DevOps and Continuous Integration/Deployment
- Agile and Scrum Methodologies
- Data Visualization
- Business Intelligence and Analytics
- Supply Chain Management and Logistics
- Sales and Negotiation Techniques
- Advanced Excel and Data Modeling
- Statistical Software Proficiency (R, SAS, SPSS)
- Cloud Computing (AWS, Azure, Google Cloud)
- Mobile Application Development
- Robotics and Automation Engineering
- Virtual Reality (VR) and Augmented Reality (AR) Development
- E-commerce Platform Management
- Digital Forensics and Incident Response
- Network Security Monitoring and Penetration Testing
- Biotechnology Techniques and Laboratory Procedures
- Geographic Information Systems (GIS) and Spatial Analysis
- Foreign Language Proficiency
- Medical Diagnosis and Patient Care
- Mechanical Engineering Design and Analysis
- Electronics Engineering and Circuit Design
- Management Consulting and Strategic Advisory

For each skill you extract, assign a percentage influence (from 0 to 100) that reflects how prominently the skill is represented in the job description.
Return ONLY the output in valid JSON format following this schema EXACTLY:
{escaped_format_instructions}

Job Description:
{{job_text}}

Your response must be valid JSON with no additional text.
"""

# Create the PromptTemplate using the variable "job_text"
template = PromptTemplate(
    input_variables=["job_text"],
    template=prompt_template,
)

# Initialize the Ollama LLM to use the local API.
llm = Ollama(model="llama3:8b", temperature=0, base_url="http://localhost:11434")

# Create an LLMChain with the prompt and output parser.
chain = LLMChain(llm=llm, prompt=template, output_parser=output_parser)

def clean_json_output(text: str) -> str:
    """
    Extracts and returns the JSON substring from the provided text.
    It finds the first '{' and the last '}' and returns everything in between.
    """
    json_match = re.search(r'(\{.*\})', text, re.DOTALL)
    if json_match:
        return json_match.group(1)
    return ""

def extract_skills(jobs_list: list, checkpoint_interval: int = 100) -> list:
    """
    Processes each job entry by extracting hard and soft skills from its job description,
    displaying a progress loader, and formatting the extracted skills as strings in the format: (skill, influence).
    
    After processing every checkpoint_interval jobs, the current job list is exported to a JSON file.
    """
    for index, job in enumerate(tqdm(jobs_list, desc="Extracting skills for jobs"), start=1):
        job_description = job.get("job_overview", "")
        if job_description:
            try:
                # Run the LLM chain on the job description.
                result = chain.run(job_text=job_description)
                # Convert the result (a Pydantic model) into a dictionary.
                skills = result.model_dump()
            except OutputParserException as e:
                # If parsing fails, extract the JSON part from the output.
                raw_output = e.llm_output if hasattr(e, "llm_output") else ""
                cleaned_output = clean_json_output(raw_output)
                try:
                    skills = output_parser.parse_result(cleaned_output).model_dump()
                except Exception:
                    skills = {"soft_skills": [], "hard_skills": []}
            # Format each skill as a string "(skill, influence)".
            formatted_soft = [f"({item['skill']}, {item['influence']})" for item in skills.get("soft_skills", [])]
            formatted_hard = [f"({item['skill']}, {item['influence']})" for item in skills.get("hard_skills", [])]
            # Update the job's extracted_skills with the formatted output.
            job["extracted_skills"] = {
                "soft_skills": formatted_soft,
                "hard_skills": formatted_hard
            }
        else:
            job["extracted_skills"] = {"soft_skills": [], "hard_skills": []}
        
        # Export checkpoint after every 'checkpoint_interval' jobs.
        if index % checkpoint_interval == 0:
            checkpoint_file = f"./data/export/jobs_with_skills_checkpoint_{index+300}.json"
            with open(checkpoint_file, "w", encoding="utf-8") as outfile:
                json.dump(jobs_list[:index], outfile, indent=2)
            print(f"Checkpoint saved: {checkpoint_file}")
    
    return jobs_list

In [ ]:
jobs_list = extract_skills(jobs_list[300:])

Extracting skills for jobs:   0%|▏                                                | 29/9346 [03:01<16:05:06,  6.22s/it]

In [22]:
df['skills.soft_skills'][0]

['(Leadership, 80)',
 '(Motivation, 70)',
 '(Communication, 90)',
 '(Teamwork, 85)']

In [19]:

# Simplify the job structure by flattening any nested dictionaries
df = pd.json_normalize(temp)
print(df.shape)
# Display the first few rows of the DataFrame
df.head()

(10, 45)


,url,company_url_overview,company_name,company_rating,job_title,job_location,job_overview,company_headquarters,company_founded_year,company_industry,...,pay_range_currency,pay_type,id,discovery_input.country,discovery_input.keyword,discovery_input.location,extracted_skills.soft_skills,extracted_skills.hard_skills,skills.soft_skills,skills.hard_skills
0,https://www.glassdoor.com/partner/jobListing.h...,https://www.ta-petro.com/,TravelCenters of America,3.0,Assistant Manager - Restaurant,"McCalla, AL",There&rsquo;s never been a better time to join...,"Westlake, OH",1972,Convenience Stores,...,USD,HOURLY,1,UK,TA Manager,Birmingham,"[{'skill': 'Communication', 'influence': 80}, ...",[],"[(Leadership, 80), (Motivation, 70), (Communic...",[]
1,https://www.glassdoor.com/partner/jobListing.h...,https://www.taservices.com/,TA Dispatch,3.3,Project Manager,"Leeds, AL",Company Description\n\n\nTA Services has been ...,"Mansfield, TX",1986,Shipping & Trucking,...,USD,ANNUAL,2,UK,TA Manager,Birmingham,"[{'skill': 'Communication', 'influence': 80}, ...","[{'skill': 'Project Management', 'influence': ...","[(Communication, 80), (Teamwork, 70), (Problem...","[(Project Management, 90), (Microsoft Office (..."
2,https://www.glassdoor.com/partner/jobListing.h...,https://www.taservices.com/,TA Dispatch,3.3,Project Manager,"Leeds, AL",Company Description\n\n\nTA Services has been ...,"Mansfield, TX",1986,Shipping & Trucking,...,USD,ANNUAL,3,UK,TA Manager,Birmingham,"[{'skill': 'Communication', 'influence': 80}, ...","[{'skill': 'Project Management', 'influence': ...","[(Communication, 80), (Teamwork, 70), (Problem...","[(Project Management, 90), (Microsoft Office (..."
3,https://www.glassdoor.com/partner/jobListing.h...,https://www.taservices.com/,TA Dispatch,3.3,Project Manager,"Leeds, AL",Company Description\n\n\nTA Services has been ...,"Mansfield, TX",1986,Shipping & Trucking,...,USD,ANNUAL,4,UK,TA Manager,Birmingham,"[{'skill': 'Communication', 'influence': 80}, ...","[{'skill': 'Project Management', 'influence': ...","[(Communication, 80), (Teamwork, 70), (Problem...","[(Project Management, 90), (Microsoft Office (..."
4,https://www.glassdoor.com/partner/jobListing.h...,https://www.taservices.com/,TA Dispatch,3.3,Project Manager,"Leeds, AL",Company Description\n\n\nTA Services has been ...,"Mansfield, TX",1986,Shipping & Trucking,...,USD,ANNUAL,5,UK,TA Manager,Birmingham,"[{'skill': 'Communication', 'influence': 80}, ...","[{'skill': 'Project Management', 'influence': ...","[(Communication, 80), (Teamwork, 70), (Problem...","[(Project Management, 90), (Microsoft Office (..."
